In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from datasets import load_dataset

ds = load_dataset("Q-b1t/IMDB-Dataset-of-50K-Movie-Reviews-Backup")

In [ ]:
# Convert the 'train' split to a Pandas DataFrame
df = pd.DataFrame(ds['train'])

# Now you can use .head()!
df.head()

In [ ]:
df.shape

In [ ]:
type(df)

In [ ]:
df.tail()

In [ ]:
df["sentiment"].value_counts()

In [ ]:
# one hot encoding
# label encoder

In [ ]:
# positive -> 1
# negative -> 0
df.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df["sentiment"].value_counts()

# LSTM -> LONG SHORT TERM MEMORY
# RNN -> TEXTUAL DATA

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
train_data, test_data = train_test_split(df, test_size = 0.2, random_state=42)

In [ ]:
train_data.shape

In [ ]:
test_data.shape

In [ ]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data["review"])

In [ ]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [ ]:
Y_train

# LSTM MODEL BUILDING

In [ ]:
model = Sequential()
model.add(Embedding(input_dim =5000, output_dim = 128, input_length = 200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout = 0.2))
model.add(Dense(1, activation = "sigmoid"))

In [ ]:
model.compile(optimizer = "adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
model.fit(X_train, Y_train, epochs = 5, batch_size = 64, validation_split = 0.2)

In [ ]:
model.summary()

In [ ]:
model.save("model.h5")

In [ ]:
import joblib
joblib.dump(tokenizer, "tokenizer.pkl")

## Model Evaluation

Now that the model is trained, let's evaluate its performance on the test dataset (`X_test`, `Y_test`). We will calculate accuracy, precision, recall, and F1-score to get a comprehensive understanding of the model's effectiveness.

In [ ]:
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the saved model (if not already in memory)
loaded_model = load_model('model.h5')

# Evaluate the model on the test data
loss, accuracy = loaded_model.evaluate(X_test, Y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Get predictions on the test set
Y_pred_probs = loaded_model.predict(X_test)
Y_pred = (Y_pred_probs > 0.5).astype(int)

# Calculate additional metrics
precision = precision_score(Y_test, Y_pred)
recall = recall_score(Y_test, Y_pred)
f1 = f1_score(Y_test, Y_pred)

print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1-Score: {f1:.4f}")

In [ ]:
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the saved model (if not already in memory)
loaded_model = load_model('model.h5')

# Evaluate the model on the test data
loss, accuracy = loaded_model.evaluate(X_test, Y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Get predictions on the test set
Y_pred_probs = loaded_model.predict(X_test)
Y_pred = (Y_pred_probs > 0.5).astype(int)

# Calculate additional metrics
precision = precision_score(Y_test, Y_pred)
recall = recall_score(Y_test, Y_pred)
f1 = f1_score(Y_test, Y_pred)

print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1-Score: {f1:.4f}")

## ULMFiT (AWD-LSTM) Implementation

To implement the ULMFiT methodology, we will use the `fastai` library, which simplifies the process of fine-tuning pre-trained language models like AWD-LSTM. The first step is to install the library.

In [ ]:
import sys
!{sys.executable} -m pip install fastai

import fastai
print(f"fastai version: {fastai.__version__}")

### Data Preparation for FastAI

Before we can train an AWD-LSTM model with `fastai`, we need to prepare our IMDb dataset in a format that `fastai` can understand. This typically involves creating a `TextDataLoaders` object from our pandas DataFrame.

In [ ]:
from fastai.text.all import *

# Create TextDataLoaders directly from the DataFrame
# text_col: The name of the column containing the text reviews
# label_col: The name of the column containing the sentiment labels
# valid_pct: Percentage of data to use for validation
# seed: For reproducibility
# bs: Batch size
# seq_len: The sequence length for padding/truncation, 72 is a common value for ULMFiT
dls_clas = TextDataLoaders.from_df(df, text_col='review', label_col='sentiment',
                              valid_pct=0.2, seed=42, bs=64, seq_len=72)

print("FastAI DataLoaders created successfully!")

# Show a batch of data to verify
dls_clas.show_batch()

## BERT-Based Transformer Model Implementation

Now, let's move on to implementing a BERT-based Transformer model for sentiment analysis. We will leverage the Hugging Face `transformers` library, which provides easy access to pre-trained BERT models and their tokenizers. This approach follows your project's goal of comparing RNNs with Transformer-based architectures.

In [ ]:
import sys
!{sys.executable} -m pip install transformers
!{sys.executable} -m pip install torch

print("Hugging Face Transformers and PyTorch installed successfully!")

### Load BERT Tokenizer and Model

We'll use the `bert-base-uncased` model, which is a popular pre-trained BERT variant. The tokenizer will preprocess our text data into a format suitable for BERT, including adding special tokens and handling padding/truncation.

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Load pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load pre-trained BERT model for sequence classification
# num_labels=2 for binary sentiment classification (positive/negative)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Set device to GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"BERT tokenizer and model loaded on device: {device}")

### Prepare Data for BERT

BERT requires specific input formats, including token IDs, attention masks, and token type IDs. We need to tokenize our `review` column and convert the `sentiment` labels to a format compatible with BERT's classification head.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split

# Use the same train_data and test_data from previous steps
# Tokenize and encode sequences from the 'review' column
encoded_data_train = tokenizer(
    train_data['review'].tolist(),
    add_special_tokens=True,
    return_attention_mask=True,
    padding='max_length',
    truncation=True,
    max_length=256,
    return_tensors='pt'
)

encoded_data_test = tokenizer(
    test_data['review'].tolist(),
    add_special_tokens=True,
    return_attention_mask=True,
    padding='max_length',
    truncation=True,
    max_length=256,
    return_tensors='pt'
)

input_ids_train = encoded_data_train['input_ids']
attention_masks_train = encoded_data_train['attention_mask']
labels_train = torch.tensor(train_data['sentiment'].tolist())

input_ids_test = encoded_data_test['input_ids']
attention_masks_test = encoded_data_test['attention_mask']
labels_test = torch.tensor(test_data['sentiment'].tolist())

# Create TensorDatasets
dataset_train = TensorDataset(input_ids_train, attention_masks_train, labels_train)
dataset_test = TensorDataset(input_ids_test, attention_masks_test, labels_test)

# Create DataLoaders
batch_size = 32 # A common batch size for BERT fine-tuning
dataloader_train = DataLoader(dataset_train, sampler=RandomSampler(dataset_train), batch_size=batch_size)
dataloader_test = DataLoader(dataset_test, sampler=SequentialSampler(dataset_test), batch_size=batch_size)

print("Data prepared for BERT fine-tuning.")
print(f"Number of training samples: {len(dataset_train)}")
print(f"Number of test samples: {len(dataset_test)}")

### Fine-tune BERT Model

We will now fine-tune the pre-trained BERT model on our sentiment analysis dataset. This involves defining training arguments and then using the `Trainer` class from Hugging Face `transformers` to manage the training loop.

In [ ]:
from torch.optim import AdamW # Corrected import for AdamW
from transformers import get_linear_schedule_with_warmup

# Optimizer & Learning Rate Scheduler
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8) # AdamW is a standard optimizer for BERT

epochs = 3 # A common number of epochs for BERT fine-tuning
total_steps = len(dataloader_train) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print("Optimizer and learning rate scheduler initialized.")

### Training Loop

We will implement a standard training loop for BERT, iterating through epochs and batches, performing forward and backward passes, and updating the model parameters.

In [ ]:
from tqdm.notebook import tqdm

# Function to calculate accuracy
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

# Training loop
for epoch_i in range(epochs):
    print(f'\n======== Epoch {epoch_i + 1} / {epochs} ========\n')
    print('Training...')

    total_loss = 0
    model.train() # Set model to training mode

    for step, batch in enumerate(dataloader_train):
        # Progress update every 40 batches.
        if step % 40 == 0 and not step == 0:
            print(f'  Batch {step:>5,}  of  {len(dataloader_train):>5,}. Loss: {total_loss/(step+1):.2f}')

        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        model.zero_grad()

        outputs = model(
            b_input_ids,
            token_type_ids=None,
            attention_mask=b_input_mask,
            labels=b_labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Clip the norm of the gradients to 1.0 to prevent "exploding gradients"

        optimizer.step()
        scheduler.step()

    avg_train_loss = total_loss / len(dataloader_train)
    print(f'\n  Average training loss: {avg_train_loss:.2f}')

    print('\nRunning Validation...')

    model.eval() # Set model to evaluation mode

    eval_loss, eval_accuracy = 0, 0
    nb_eval_steps, nb_eval_examples = 0, 0

    for batch in dataloader_test:
        batch = tuple(t.to(device) for t in batch)

        b_input_ids, b_input_mask, b_labels = batch

        with torch.no_grad(): # Disable gradients for validation
            outputs = model(
                b_input_ids,
                token_type_ids=None,
                attention_mask=b_input_mask
            )

        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        tmp_eval_accuracy = flat_accuracy(logits, label_ids)
        eval_accuracy += tmp_eval_accuracy
        nb_eval_steps += 1

    print(f'  Validation Accuracy: {eval_accuracy/nb_eval_steps:.4f}')

print('\nTraining complete!')

# Save the fine-tuned model
model.save_pretrained('./bert_sentiment_model')
tokenizer.save_pretrained('./bert_sentiment_model')
print("BERT model and tokenizer saved.")

In [ ]:
# This cell is no longer needed as saving is handled in df8dc9ed.

### Verify Saved Model Files

Let's check the contents of the directory where the model and tokenizer were supposed to be saved. This will help confirm that the `save_pretrained` calls worked as expected and that the files are present locally.

In [ ]:
import os

print("Contents of current directory:")
!ls -F

print("\nContents of ./bert_sentiment_model/:")
!ls -F ./bert_sentiment_model/

Please run the above cell. Once you confirm the `bert_sentiment_model` directory contains files like `pytorch_model.bin` (or similar model weights) and `vocab.txt`/`tokenizer.json`, then re-execute **cell `740c5963`**. If the files are there and you still get the error, we might need to explicitly tell `from_pretrained` to look only for local files.

Please execute the above cell first. After confirming the directory is created and the model saved, you can then re-execute the cells below starting from **cell `740c5963`** to correctly load and use your model. The previous error should now be resolved as the directory will explicitly exist.

### Evaluate BERT Model

### Create and Fine-tune the Language Model Learner

Now we'll create a `text_classifier_learner` using the `AWD_LSTM` architecture and our prepared `dls_clas`. We'll then fine-tune it on our dataset. This involves several steps: finding a good learning rate, fitting the model, and then gradually unfreezing layers for further fine-tuning.

In [ ]:
from fastai.text.all import *

# Create a text classifier learner
# arch: The architecture to use, here it's AWD_LSTM for ULMFiT
# dls: The DataLoaders created previously
# metrics: The evaluation metrics, here accuracy
learn = text_classifier_learner(dls_clas, AWD_LSTM, metrics=accuracy).to_fp16()

print("Language model learner created successfully!")

### Train the Classifier Head

Before fine-tuning the entire model, we'll train only the classifier head while keeping the pre-trained language model layers frozen. This helps the classifier adapt to our specific task without altering the robust pre-trained embeddings.

In [ ]:
# Find an optimal learning rate
lr_finder = learn.lr_find()

# Fit the head of the model for one epoch using the found learning rate
# This trains only the new classification layers added on top of the pre-trained model
learn.fit_one_cycle(1, lr_finder.valley)

print("Classifier head trained for one epoch.")

### Fine-tune the Entire Model (Unfreeze and Train More)

Next, we'll unfreeze more layers and fine-tune the entire model with discriminative learning rates. This allows the pre-trained weights to be slightly adjusted for our specific dataset and task, leading to better performance.

In [ ]:
# Unfreeze the last few layers and train for a few epochs with discriminative learning rates
learn.unfreeze()
learn.fit_one_cycle(3, slice(1e-3/(2.6**4),1e-3))

print("Model fine-tuned with unfreezing.")

# You can unfreeze more layers or the entire model and train for more epochs if needed
# learn.unfreeze()
# learn.fit_one_cycle(5, slice(1e-3/(2.6**4),1e-3))


### Evaluate ULMFiT Model

After fine-tuning, let's evaluate the performance of the ULMFiT model on the validation set. We'll check the accuracy and other relevant metrics.

In [ ]:
# Get predictions and actual labels from the validation set
preds, targs = learn.get_preds(dl=dls_clas.valid)

# Calculate accuracy on the validation set
acc = accuracy(preds, targs)
print(f"ULMFiT Model Accuracy on Validation Set: {acc.item():.4f}")

# You can also get other metrics using sklearn after converting to numpy arrays
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

# Convert predictions and targets to CPU for scikit-learn
preds_cpu = preds.argmax(dim=1).cpu().numpy()
targs_cpu = targs.cpu().numpy()

print("\nClassification Report:")
print(classification_report(targs_cpu, preds_cpu))

print(f"Precision: {precision_score(targs_cpu, preds_cpu):.4f}")
print(f"Recall: {recall_score(targs_cpu, preds_cpu):.4f}")
print(f"F1-Score: {f1_score(targs_cpu, preds_cpu):.4f}")

### Create and Fine-tune the Language Model Learner

Now we'll create a `text_classifier_learner` using the `AWD_LSTM` architecture and our prepared `dls_clas`. We'll then fine-tune it on our dataset. This involves several steps: finding a good learning rate, fitting the model, and then gradually unfreezing layers for further fine-tuning.

In [ ]:
from fastai.text.all import *

# Create a text classifier learner
# arch: The architecture to use, here it's AWD_LSTM for ULMFiT
# dls: The DataLoaders created previously
# metrics: The evaluation metrics, here accuracy
learn = text_classifier_learner(dls_clas, AWD_LSTM, metrics=accuracy).to_fp16()

print("Language model learner created successfully!")

### Train the Classifier Head

Before fine-tuning the entire model, we'll train only the classifier head while keeping the pre-trained language model layers frozen. This helps the classifier adapt to our specific task without altering the robust pre-trained embeddings.

In [ ]:
# Find an optimal learning rate
lr_finder = learn.lr_find()

# Fit the head of the model for one epoch
# This trains only the new classification layers added on top of the pre-trained model
learn.fit_one_cycle(1, lr_finder.valley)

### Fine-tune the Entire Model

Next, we'll unfreeze more layers and fine-tune the entire model with discriminative learning rates. This allows the pre-trained weights to be slightly adjusted for our specific dataset and task, leading to better performance.

In [ ]:
# Unfreeze the last few layers and train for a few epochs
learn.unfreeze()
learn.fit_one_cycle(3, slice(1e-3/(2.6**4),1e-3))

# You can unfreeze more layers or the entire model and train for more epochs if needed
# learn.unfreeze()
# learn.fit_one_cycle(5, slice(1e-3/(2.6**4),1e-3))

### Evaluate ULMFiT Model

After fine-tuning, let's evaluate the performance of the ULMFiT model on the validation set. We'll check the accuracy.

In [ ]:
# Get predictions and actual labels
preds, targs = learn.get_preds()

# Calculate accuracy on the validation set
acc = accuracy(preds, targs)
print(f"ULMFiT Model Accuracy on Validation Set: {acc.item():.4f}")

# You can also get other metrics using sklearn after converting to numpy arrays
# from sklearn.metrics import classification_report
# print(classification_report(targs.numpy(), preds.argmax(dim=1).numpy()))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Put the model in evaluation mode
model.eval()

predictions = []
true_labels = []

# Predict on test batches
for batch in dataloader_test:
    batch = tuple(t.to(device) for t in batch)
    b_input_ids, b_input_mask, b_labels = batch

    with torch.no_grad():
        outputs = model(
            b_input_ids,
            token_type_ids=None,
            attention_mask=b_input_mask
        )

    logits = outputs.logits
    logits = logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    predictions.append(logits)
    true_labels.append(label_ids)

# Concatenate the predictions and true labels
predictions = np.concatenate(predictions, axis=0)
true_labels = np.concatenate(true_labels, axis=0)

# Get predicted class labels (0 or 1)
predicted_classes = np.argmax(predictions, axis=1).flatten()

# Calculate metrics
bert_accuracy = accuracy_score(true_labels, predicted_classes)
bert_precision = precision_score(true_labels, predicted_classes)
bert_recall = recall_score(true_labels, predicted_classes)
bert_f1 = f1_score(true_labels, predicted_classes)

print(f"BERT Test Accuracy: {bert_accuracy:.4f}")
print(f"BERT Test Precision: {bert_precision:.4f}")
print(f"BERT Test Recall: {bert_recall:.4f}")
print(f"BERT Test F1-Score: {bert_f1:.4f}")

## Comparative Analysis

In [ ]:
import pandas as pd

# Define the metrics for each model
custom_lstm_metrics = {
    'Model': 'Custom LSTM',
    'Accuracy': 0.8711,
    'Precision': 0.8384,
    'Recall': 0.9218,
    'F1-Score': 0.8782
}

ulmfit_metrics = {
    'Model': 'ULMFiT (AWD-LSTM)',
    'Accuracy': 0.9186,
    'Precision': 0.9107,
    'Recall': 0.9272,
    'F1-Score': 0.9189
}

bert_metrics = {
    'Model': 'BERT',
    'Accuracy': 0.9255,
    'Precision': 0.9202,
    'Recall': 0.9331,
    'F1-Score': 0.9266
}

# Create a DataFrame to display the comparison
comparison_df = pd.DataFrame([custom_lstm_metrics, ulmfit_metrics, bert_metrics])

display(comparison_df.set_index('Model'))

### Visualizing Comparative Metrics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Melt the DataFrame to a long format for easier plotting with seaborn
metrics_melted = comparison_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 7))
sns.barplot(x='Metric', y='Score', hue='Model', data=metrics_melted, palette='viridis')
plt.title('Comparative Analysis of Model Performance', fontsize=16)
plt.xlabel('Metrics', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.ylim(0.8, 1.0) # Set y-axis limits to focus on the performance range
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## Final Project Documentation: Sentiment Analysis Model Comparison

### Project Objective

The primary objective of this project was to compare the performance of three distinct neural network architectures—a Custom LSTM, a pre-trained AWD-LSTM (ULMFiT), and a BERT-based Transformer model—on the IMDb movie reviews sentiment analysis dataset. The goal was to understand their relative strengths, convergence characteristics, and generalization capabilities for binary sentiment classification.

### Model Implementations and Key Findings

#### 1. Custom LSTM Model

*   **Implementation:** A sequential Keras model with an Embedding layer, an LSTM layer, and a Dense sigmoid output layer. The text data was tokenized and padded to a fixed sequence length.
*   **Training:** Trained for 5 epochs with Adam optimizer and binary cross-entropy loss.
*   **Evaluation Metrics:**
    *   Accuracy: 0.8711
    *   Precision: 0.8384
    *   Recall: 0.9218
    *   F1-Score: 0.8782
*   **Summary:** The custom LSTM model provided a solid baseline performance, demonstrating its ability to capture sequential dependencies in text for sentiment classification.

#### 2. ULMFiT (AWD-LSTM) Model

*   **Implementation:** Utilized the `fastai` library to fine-tune a pre-trained AWD-LSTM language model. Data was prepared using `TextDataLoaders.from_df`.
*   **Training:** Involved a two-stage fine-tuning process: training the classifier head first, followed by gradually unfreezing and fine-tuning the entire model with discriminative learning rates.
*   **Evaluation Metrics (on Validation Set):**
    *   Accuracy: 0.9186
    *   Precision: 0.9107
    *   Recall: 0.9272
    *   F1-Score: 0.9189
*   **Summary:** ULMFiT significantly outperformed the custom LSTM, showcasing the benefits of transfer learning and fine-tuning a powerful pre-trained language model on a new task.

#### 3. BERT-Based Transformer Model

*   **Implementation:** Leveraged the Hugging Face `transformers` library, using `BertTokenizer` and `BertForSequenceClassification` (`bert-base-uncased`). Data was tokenized and formatted with attention masks and input IDs suitable for BERT.
*   **Training:** Fine-tuned for 3 epochs using the AdamW optimizer and a linear learning rate scheduler with warm-up. Training included an evaluation step per epoch.
*   **Evaluation Metrics (on Test Set):**
    *   Accuracy: 0.9255
    *   Precision: 0.9202
    *   Recall: 0.9331
    *   F1-Score: 0.9266
*   **Summary:** The BERT model achieved the highest performance among all three, confirming the state-of-the-art capabilities of Transformer architectures for natural language understanding tasks, particularly when fine-tuned on specific datasets.

### Comparative Analysis and Conclusion

The table and bar chart below summarize the performance of all three models across key sentiment analysis metrics:



In [ ]:
display(comparison_df.set_index('Model'))

The comparative analysis clearly indicates that the BERT-based Transformer model achieved the best overall performance, closely followed by the ULMFiT (AWD-LSTM) model. Both pre-trained models demonstrated superior accuracy, precision, recall, and F1-score compared to the custom-built LSTM model. This highlights the effectiveness of leveraging large pre-trained language models and transfer learning for achieving high performance in downstream NLP tasks like sentiment analysis.

This project successfully demonstrated the progression in NLP model effectiveness, from traditional RNNs to advanced Transformer architectures, providing a clear comparison of their capabilities on the IMDb movie reviews dataset.